# Verify `exist_shot` and `database.load` (TS + CX)

Checklist notebook for the changes made on 2026-07-21:

1. **`exist_shot` lists TS and CX shots** (top of this notebook).
2. **`load(representation="imas", paths=...)` routes to the native IDS loader.**
3. **Re-ran the Thomson workflow for shot 39915** — it was stale `invalid`; now `core_profile`.
4. **`load(shot, representation="imas", paths="thomson_scattering"/"charge_exchange")` works**, and
   fails with a clear message when that IDS was never stored for the shot.

Bugs fixed along the way:
- `vaft.database.utils.PROCESSED_H5_PATH` read `public_omas/processed_shots.h5` while the updaters
  write `public/processed_shots.h5` — the reader never saw reprocessing. Now aligned to `public/`.
- `save_processed_shot` used `np.string_` (removed in NumPy 2.0) → `np.bytes_`.
- `database.load(representation="imas", paths=...)` for an IDS not on HSDS reports an actionable error;
  now raises a clear `FileNotFoundError` listing the available IDS.

## 1. List shots with processed diagnostics

`vaft.database.exist_shot(data_filter=...)` reads the corrective-pipeline registry
(`processed_shots.h5`). `Status` is a processing-attempt log — drop `invalid` for shots that
were actually processed. Requires an HSDS connection (`hsconfigure`).

In [ ]:
import numpy as np
import vaft
from vaft import database

# Thomson scattering shots
ts_df = vaft.database.exist_shot(data_filter='ts')          # alias: 'thomson_scattering'

# Charge exchange (IDS/CES) shots
cx_df = vaft.database.exist_shot(data_filter='cx')          # alias: 'charge_exchange'

if ts_df is not None:
    have_ts = ts_df[ts_df['Status'] != 'invalid']['Shot Number'].tolist()
    print(f"\n{len(have_ts)} shots with usable Thomson data (non-invalid).")
print("charge_exchange shots recorded:", 0 if cx_df is None else len(cx_df))

## 2. `load` dispatch: explicit representation

- Default `representation="omas"` -> OMAS **ODS**.
- `representation="imas", paths=...` -> native **IDS** via `imas.DBEntry`.
- `imas_version` is optional and inferred from the remote metadata when available.

In [ ]:
# Native requests require a top-level IDS path
try:
    database.load(shot=39915, representation="imas")
except ValueError as e:
    print("ValueError (expected):", e)

## 3. Load TS + core_profiles for shot 39915

39915 was reprocessed, so `thomson_scattering.h5` and `core_profiles.h5` are now on HSDS.

In [ ]:
ts = database.load(shot=39915, source="public", representation="imas", paths="thomson_scattering")
print("thomson_scattering.time (ms):", np.asarray(ts.time) * 1e3)
print("n channels:", len(ts.channel))

cp = database.load(shot=39915, source="public", representation="imas", paths="core_profiles")
print("core_profiles slices:", len(cp.profiles_1d))

## 4. Requesting an IDS that was never stored -> clear error

Most shots (and every shot for `charge_exchange`, since no CX images are on HSDS yet) do not have
these IDS uploaded. The loader now says so explicitly instead of crashing on `hsget`.

In [ ]:
for shot, ids_name in [(39513, "thomson_scattering"), (39513, "charge_exchange")]:
    try:
        database.load(shot=shot, source="public", representation="imas", paths=ids_name)
    except FileNotFoundError as e:
        print(f"[{shot} / {ids_name}] ->", e)

## 5. How 39915 was fixed (reference — writes to production HSDS)

39915 was `invalid` because an early (2024-05-14) pipeline run failed before saving; the Thomson
`.mat` itself is fine. Re-running the per-shot workflow maps Thomson, fits `core_profiles` against
the CHEASE geqdsk, and uploads the IDS images to `/public/39915/`.

**This cell writes to the shared database** — it re-uploads every IDS image for the shot and updates
the registry. It is guarded by `RUN_WORKFLOW = False`; flip it to re-run for another shot.

In [ ]:
RUN_WORKFLOW = False  # set True to actually reprocess (writes to HSDS)

if RUN_WORKFLOW:
    import os, sys
    from datetime import datetime
    sys.path.insert(0, "../workflow/automatic_pipeline_2_corrective_data_update")
    import update_thomson_scattering_and_core_profile as wf

    shot = 39915
    mat = os.path.abspath("../vaft/data/legacy/NeTe_Shot39915_v9_rev.mat")
    ods, shot = wf.update_thomson_auto(mat)                     # map TS + save ODS to HSDS
    fitted = wf.fit_thomson_profile_auto_all_times(ods, shot)  # fit core_profiles (needs CHEASE)
    status = "core_profile" if fitted else "thomson_only"
    wf.save_processed_shot(shot, datetime.now().isoformat(), status=status)
    print(f"reprocessed {shot}: status={status}")
else:
    print("RUN_WORKFLOW is False - skipping the production write.")